# 06 — Model Replacement Analysis

## Question

Can a cheaper model safely replace a more expensive one on a real workload, measured
by cost per successful task?

**This notebook cannot answer that question, and it does not try.** The dataset the
analysis requires is not published. Everything below documents that finding and the
evidence for it.


## Dataset

**Required dataset: a model-replacement result set — paired per-task success and
cost outcomes for a current model and a candidate model on a shared task set.**

**Status: not published.**

The repository `harpd-dev/model-replacement-benchmark` exists, but it contains no
result data. Its contents are:

| Path | What it is |
|---|---|
| `README.md` | Usage documentation |
| `compare.mjs` | A comparison script |
| `LICENSE` | MIT licence |

`compare.mjs` computes its numbers from a hard-coded price table and a stub:

```js
// Stub: replace with a real provider call. Here we just parse the task text as a stand-in.
function run(model) {
  let ok = 0
  for (const t of tasks) {
    const text = `[stub: call ${model} with "${t.input}"]`
    if (score(parse(text), t.keys)) ok++
  }
  ...
  const success = ok / tasks.length || 0.9
```

No provider is called, and when the stub scores nothing it falls back to a default
success rate of `0.9`. Any number produced by that path would be an artefact of the
script, not a measurement.

This notebook therefore stops at the probe. It does not load, substitute, simulate
or estimate a replacement result set.


## Method

1. Probe the `model-replacement-benchmark` repository for any published result
   dataset (`loader.model_replacement_status`).
2. Load Harpd's own evidence register and read the entries under `notClaimed` and
   `rules` that govern this exact claim.
3. Report the outcome. If no dataset is found, stop — do not compute.

The probe is the analysis. Its result is the finding.


## Analysis

In [1]:
import os
import sys
from pathlib import Path

# Make the repository root importable no matter where Jupyter was started.
ROOT = Path.cwd()
while not (ROOT / "harpd_research").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from harpd_research import analysis, loader

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)

print("repo root :", ROOT)
print("data base :", loader.resolved_data_base())
print("pandas    :", pd.__version__)
print()
print("DOMAIN CAVEAT")
print(analysis.PLACEMENT_CAVEAT)


repo root : /Users/shankou/harpd/harpd-ai-research-notebooks
data base : https://raw.githubusercontent.com/harpd-dev/harpd-ai-datasets/main/
pandas    : 3.0.5

DOMAIN CAVEAT
Rank Points are promotional placement bought with Credits — NOT an editorial quality score. Any ordering below is a promotional placement ordering, not a ranking of product quality.


In [2]:
status = loader.model_replacement_status()

print("probed repository :", status["repo"])
print("data files found  :", status["data_files"] or "(none)")
print("dataset available :", status["dataset_available"])
print()
print("reason:")
print(" ", status["reason"])

probed repository : https://github.com/harpd-dev/model-replacement-benchmark
data files found  : (none)
dataset available : False

reason:
  No model-replacement result dataset is published. The repository ships compare.mjs, whose provider calls are an explicit stub, and Harpd's own evidence register lists cross-model switch safety as notClaimed.


In [3]:
# Read the publisher's own position on this claim, from the evidence register.
evidence = loader.load_evidence()

print("Evidence register chain:", " -> ".join(evidence["chain"]))
print()
print("Rules that bind any model-replacement claim:")
for rule in evidence["rules"]:
    print(f"  - {rule}")

print()
print("Statements the publisher explicitly does NOT claim:")
for item in evidence["notClaimed"]:
    print(f"  - NOT CLAIMED: {item['statement']}")
    print(f"    why: {item['why']}")
    print(f"    methodology: {item.get('methodologyUrl')}")
    print()

Evidence register chain: CLAIM -> EVIDENCE -> DATASET -> METHODOLOGY -> SOURCE -> TIMESTAMP

Rules that bind any model-replacement claim:
  - A claim is publishable only when it resolves to a registered dataset, a methodology URL and a real data timestamp.
  - MODELED claims must be labelled as estimates wherever they appear and are never presented as measured results.
  - EDITORIAL and USER_SUBMITTED claims never appear on a fact surface; user-submitted content must be attributed.
  - Harpd Rank Points are promotional placement, not an editorial quality score, and may not be described with editorial-quality wording.
  - Nothing here is back-filled: a missing dataset or timestamp produces a violation, not a default value.

Statements the publisher explicitly does NOT claim:
  - NOT CLAIMED: That any model is "safe to switch to" for a production workload.
    why: A switch decision requires a real-task quality gate on the specific workload. No dataset covers that yet.
    methodology: h

In [4]:
# Search the evidence register for any claim about model replacement.
claims = evidence["claims"]
keywords = ("replace", "switch", "cheaper", "migration")
matches = [
    c for c in claims
    if any(k in (c.get("claim", "") + " " + c.get("id", "")).lower() for k in keywords)
]

print(f"claims in register      : {len(claims)}")
print(f"claims mentioning replacement/switch/cheaper: {len(matches)}")
print()
if matches:
    for c in matches:
        print(f"  - {c['id']}: {c['claim'][:120]}")
else:
    print("No claim in the register asserts that any model can be safely replaced")
    print("by a cheaper one.")

print()
display(analysis.evidence_audit(evidence))

claims in register      : 5
claims mentioning replacement/switch/cheaper: 0

No claim in the register asserts that any model can be safely replaced
by a cheaper one.



,claimType,claims,supported,withSource
0,OBSERVED,4,4,4
1,MODELED,1,1,1


In [5]:
# Related datasets that DO exist, and why they are not a substitute.
benchmark, meta = loader.load_benchmark_results()

print("A related dataset does exist: llm-cost-benchmark")
print("  models covered     :", len(benchmark))
print("  isModeled          :", meta.get("isModeled"))
print("  methodologyNote    :", meta.get("methodologyNote"))
print()
print("Why it is NOT a substitute for this notebook's question:")
print("  1. It measures models on ONE task family (JSON extraction), not on the")
print("     reader's workload. A switch decision is workload-specific.")
print("  2. It reports per-model cost and success, not a paired per-task outcome")
print("     for a current/candidate model pair.")
print("  3. Its own metadata labels the figures a MODELED estimate, not a live run.")
print()
print("Notebook 07 analyses that dataset on its own terms. This notebook does not")
print("borrow its numbers to answer a question it cannot answer.")

A related dataset does exist: llm-cost-benchmark
  models covered     : 8
  isModeled          : True
  methodologyNote    : MODELED ESTIMATE. Prices are public 2026 list prices; success/latency are assumed from published 2025–2026 behavior. Replace with live runs via the open-source runner (node runner/index.mjs) before citing.

Why it is NOT a substitute for this notebook's question:
  1. It measures models on ONE task family (JSON extraction), not on the
     reader's workload. A switch decision is workload-specific.
  2. It reports per-model cost and success, not a paired per-task outcome
     for a current/candidate model pair.
  3. Its own metadata labels the figures a MODELED estimate, not a live run.

Notebook 07 analyses that dataset on its own terms. This notebook does not
borrow its numbers to answer a question it cannot answer.


## Visualization

No chart is produced. Plotting anything here would require values that do not
exist, and a chart is the fastest way to launder an invented number into an
authoritative-looking figure.


In [6]:
produced_chart = False

print("Chart produced:", produced_chart)
print()
print("Reason: a model-replacement comparison chart needs paired per-task success")
print("and cost outcomes for a current/candidate model pair. No such dataset is")
print("published, so there is nothing to plot.")
print()
print("This cell is intentionally empty of data. It exists so that the absence is")
print("visible in the notebook rather than silently skipped.")

Chart produced: False

Reason: a model-replacement comparison chart needs paired per-task success
and cost outcomes for a current/candidate model pair. No such dataset is
published, so there is nothing to plot.

This cell is intentionally empty of data. It exists so that the absence is
visible in the notebook rather than silently skipped.


## Findings

The finding is an absence, stated plainly.


In [7]:
findings = [
    "The dataset required to answer this question is NOT published. "
    "No model-replacement result set exists in harpd-dev/model-replacement-benchmark.",

    "The only artefact in that repository is compare.mjs, whose provider calls are "
    "an explicit stub with a hard-coded fallback success rate of 0.9. Any figure it "
    "emits is a property of the script, not a measurement.",

    "Harpd's own evidence register lists this claim under notClaimed: 'That any "
    "model is safe to switch to for a production workload', with the stated reason "
    "that a switch decision requires a real-task quality gate on the specific "
    "workload and that no dataset covers it yet.",

    "The register's rules also state that MODELED claims must be labelled as "
    "estimates wherever they appear and are never presented as measured results, "
    "and that a missing dataset produces a violation rather than a default value.",

    "No number is reported for model replacement. No success rate, no cost saving "
    "and no recommendation is offered, because none can be computed from real data.",

    "A partial substitute exists (llm-cost-benchmark) but it answers a different "
    "question: single-model cost on one task family, labelled MODELED. It is "
    "analysed in notebook 07 and is not used here.",
]

for i, line in enumerate(findings, 1):
    print(f"{i}. {line}")

1. The dataset required to answer this question is NOT published. No model-replacement result set exists in harpd-dev/model-replacement-benchmark.
2. The only artefact in that repository is compare.mjs, whose provider calls are an explicit stub with a hard-coded fallback success rate of 0.9. Any figure it emits is a property of the script, not a measurement.
3. Harpd's own evidence register lists this claim under notClaimed: 'That any model is safe to switch to for a production workload', with the stated reason that a switch decision requires a real-task quality gate on the specific workload and that no dataset covers it yet.
4. The register's rules also state that MODELED claims must be labelled as estimates wherever they appear and are never presented as measured results, and that a missing dataset produces a violation rather than a default value.
5. No number is reported for model replacement. No success rate, no cost saving and no recommendation is offered, because none can be comp

## Limitations

- **The primary limitation is that there is no data.** This is not a caveat about
  method; it is the result.
- A switch decision is workload-specific. Even a complete public benchmark could
  not decide replacement for a particular reader's workload.
- `compare.mjs` is a usable *tool* — a reader with provider API keys can generate
  their own paired results — but a tool is not a dataset, and this repository does
  not report tool output as a finding.
- The evidence register's `notClaimed` entry is the publisher's own statement, not
  an independent audit. It is cited as the publisher's position.
- No inference is drawn about whether such a replacement would be safe. The
  question is left open deliberately.


## Reproducibility

In [8]:
import platform
import sys

print("Reproduce this notebook")
print("=" * 60)
print("python :", sys.version.split()[0], "on", platform.platform())
print("pandas :", pd.__version__)
print("data   :", loader.resolved_data_base())
print()
print("From a clean machine:")
print("  git clone https://github.com/harpd-dev/harpd-ai-research-notebooks")
print("  cd harpd-ai-research-notebooks")
print("  python -m venv .venv && . .venv/bin/activate")
print("  pip install -r requirements.txt")
print("  jupyter nbconvert --to notebook --execute --inplace notebooks/<this file>")
print()
print("Offline / pinned snapshot:")
print("  HARPD_DATA_BASE=/path/to/harpd-ai-datasets/ jupyter nbconvert ...")

print()
print("Reproducing this notebook's finding:")
print("  1. gh repo clone harpd-dev/model-replacement-benchmark /tmp/mrb -- --depth 1")
print("  2. ls -R /tmp/mrb        # no data/ directory, no result files")
print("  3. grep -n 'Stub' /tmp/mrb/compare.mjs")
print("  4. Re-run this notebook: loader.model_replacement_status() returns")
print("     dataset_available=False")


Reproduce this notebook
python : 3.13.12 on macOS-26.6.2-arm64-arm-64bit-Mach-O
pandas : 3.0.5
data   : https://raw.githubusercontent.com/harpd-dev/harpd-ai-datasets/main/

From a clean machine:
  git clone https://github.com/harpd-dev/harpd-ai-research-notebooks
  cd harpd-ai-research-notebooks
  python -m venv .venv && . .venv/bin/activate
  pip install -r requirements.txt
  jupyter nbconvert --to notebook --execute --inplace notebooks/<this file>

Offline / pinned snapshot:
  HARPD_DATA_BASE=/path/to/harpd-ai-datasets/ jupyter nbconvert ...

Reproducing this notebook's finding:
  1. gh repo clone harpd-dev/model-replacement-benchmark /tmp/mrb -- --depth 1
  2. ls -R /tmp/mrb        # no data/ directory, no result files
  3. grep -n 'Stub' /tmp/mrb/compare.mjs
  4. Re-run this notebook: loader.model_replacement_status() returns
     dataset_available=False


## Sources

- Probed repository: <https://github.com/harpd-dev/model-replacement-benchmark>
- Related (different) dataset: <https://github.com/harpd-dev/llm-cost-benchmark>
- Evidence register: `data/evidence/evidence.json`
  (<https://github.com/harpd-dev/harpd-ai-datasets>)
- Publisher's model-replacement methodology: <https://harpd.com/methodology/model-replacement/>
- Canonical data home: <https://harpd.com/data/>
- Licence: dataset content CC BY 4.0; `model-replacement-benchmark` code MIT.

**Licence.** All Harpd dataset content is published under CC BY 4.0.

**Attribution.** Harpd (<https://harpd.com>).

**Not a source for.** Product quality, traffic, revenue, review scores or
market size. None of those are measured by these datasets, and none are
estimated in this notebook.
